<a href="https://colab.research.google.com/github/ntwdtlari/AULA-INTELIGENCIA-ARTIFICIAL/blob/main/Aula_06_Chat_PDF_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📄 Aula 06 — Chat com PDF

Pipeline RAG usando LangChain

**Fluxo:**
1. Carregar PDF
2. Quebrar em chunks
3. Gerar embeddings
4. Guardar no banco vetorial (Chroma)
5. Retriever busca os trechos relevantes
6. LLM responde com base no contexto

In [1]:
# Instalação das bibliotecas
!pip install -q \
    langchain-community \
    langchain-text-splitters \
    langchain-huggingface \
    langchain-core \
    chromadb \
    sentence-transformers \
    transformers \
    pypdf \
    huggingface_hub \
    accelerate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 33.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 54.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7/333.7 kB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 46.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 41.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 67.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.

In [2]:
# Imports
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from transformers import pipeline

print("Imports OK!")

Imports OK!


In [7]:
# @title
# 1. Carregar PDF
# Faça upload do seu PDF no Colab antes de executar esta célula
arquivo = '/content/pdf_1772.pdf'

loader = PyPDFLoader(arquivo)
documents = loader.load()



In [8]:
print(f"Total de páginas carregadas: {len(documents)}")


Total de páginas carregadas: 5


In [9]:
print("\nPrimeira página (trecho):")



Primeira página (trecho):


In [17]:
print(documents[0].page_content[:500])

Gestão de Pessoas e Psicologia Organizacional e do Trabalho
Se você deseja impulsionar sua carreira no universo corporativo, se tornar um profissional altamente capacitado e desenvolver uma
visão estratégica sobre a gestão de pessoas, a nossa especialização em Gestão de Pessoas e Psicologia Organizacional e do
Trabalho é a oportunidade que você estava esperando. 
Com uma formação focada em resultados práticos e soluções inovadoras, este curso é ideal para quem quer se destacar no
mercado de trab


In [11]:
# 2. Quebrar texto em chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=80
)
docs = text_splitter.split_documents(documents)

print(f"Total de chunks gerados: {len(docs)}")

Total de chunks gerados: 57


In [18]:
# 3. Embeddings + Banco vetorial + Retriever
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

db = Chroma.from_documents(docs, embeddings)

retriever = db.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 3}
)

print("Banco vetorial e retriever prontos!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Banco vetorial e retriever prontos!


In [19]:
# 4. LLM local

pipe = pipeline(
    "text-generation",
    model="google/flan-t5-base",
    max_new_tokens=200,
)

llm = HuggingFacePipeline(pipeline=pipe)

print("Modelo LLM carregado!")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

...

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLlamaF

Modelo LLM carregado!


In [20]:
# 5. Prompt + Chain

prompt = PromptTemplate.from_template("""\
Você é um assistente especializado em análise de documentos.
Responda de forma clara e objetiva com base apenas no contexto abaixo.
Se não encontrar a resposta, diga: "Não encontrei essa informação no documento."

Contexto:
{context}

Pergunta: {question}

Resposta:""")

def formatar_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Pipeline LCEL: recupera → formata → prompt → llm → texto
chain = (
    {"context": retriever | formatar_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("Ok!")

Ok!


In [21]:
# 6. Loop de perguntas
print("Chat com PDF iniciado! Digite 'sair' para encerrar.\n")

while True:
    pergunta = input("Pergunta: ")
    if pergunta.lower() == "sair":
        print("Encerrando...")
        break

    resposta = chain.invoke(pergunta)
    print("\nResposta:", resposta)
    print("-" * 60)

Chat com PDF iniciado! Digite 'sair' para encerrar.

Pergunta: o que é gestão de pessoas?


Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Resposta: Você é um assistente especializado em análise de documentos.
Responda de forma clara e objetiva com base apenas no contexto abaixo.
Se não encontrar a resposta, diga: "Não encontrei essa informação no documento."

Contexto:
visão estratégica sobre a gestão de pessoas, a nossa especialização em Gestão de Pessoas e Psicologia Organizacional e do
Trabalho é a oportunidade que você estava esperando. 
Com uma formação focada em resultados práticos e soluções inovadoras, este curso é ideal para quem quer se destacar no

práticas e ferramentas para a resolução eficaz de conflitos e a gestão de erros no contexto de gestão de pessoas.
E-mail:
pos.toledo@pucpr.br
Telefone:
45991549135 www.pucpr.br

Gestão de Pessoas: Análise de Dados por Meio do People Analytics
Esta disciplina tem como objetivo capacitar os alunos a compreender e aplicar as técnicas e princípios do People
Analytics na gestão de recursos humanos. Os alunos serão preparados para coletar, analisar e interpretar dados

P

Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Resposta: Você é um assistente especializado em análise de documentos.
Responda de forma clara e objetiva com base apenas no contexto abaixo.
Se não encontrar a resposta, diga: "Não encontrei essa informação no documento."

Contexto:
visão estratégica sobre a gestão de pessoas, a nossa especialização em Gestão de Pessoas e Psicologia Organizacional e do
Trabalho é a oportunidade que você estava esperando. 
Com uma formação focada em resultados práticos e soluções inovadoras, este curso é ideal para quem quer se destacar no

práticas e ferramentas para a resolução eficaz de conflitos e a gestão de erros no contexto de gestão de pessoas.
E-mail:
pos.toledo@pucpr.br
Telefone:
45991549135 www.pucpr.br

Gestão de Pessoas: Análise de Dados por Meio do People Analytics
Esta disciplina tem como objetivo capacitar os alunos a compreender e aplicar as técnicas e princípios do People
Analytics na gestão de recursos humanos. Os alunos serão preparados para coletar, analisar e interpretar dados

P

KeyboardInterrupt: Interrupted by user

---
## Groq (gratuito)

1. Crie sua chave gratuita em: https://console.groq.com
2. E teste

In [22]:
# Testando o GROK
!pip install -q langchain-groq

import os
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Cole aqui sua chave Groq
os.environ["GROQ_API_KEY"] = ""

llm_groq = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

prompt = PromptTemplate.from_template("""\
Você é um assistente especializado em análise de documentos.
Responda de forma clara e objetiva com base apenas no contexto abaixo.
Se não encontrar a resposta, diga: "Não encontrei essa informação no documento."

Contexto:
{context}

Pergunta: {question}

Resposta:""")

def formatar_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

chain = (
    {"context": retriever | formatar_docs, "question": RunnablePassthrough()}
    | prompt
    | llm_groq
    | StrOutputParser()
)

print("Chain com Groq pronta! Digite 'sair' para encerrar.\n")

while True:
    pergunta = input("Pergunta: ")
    if pergunta.lower() == "sair":
        print("Encerrando...")
        break

    resposta = chain.invoke(pergunta)
    print("\nResposta:", resposta)
    print("-" * 60)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 4.3 MB/s eta 0:00:00
Chain com Groq pronta! Digite 'sair' para encerrar.

Pergunta: o que é gestão de pessoas?

Resposta: Não encontrei uma definição explícita de "gestão de pessoas" no documento. No entanto, é possível inferir que a gestão de pessoas está relacionada à administração de recursos humanos, incluindo a resolução de conflitos, gestão de erros e análise de dados por meio do People Analytics.
------------------------------------------------------------
Pergunta: como posso impulssionar minha carreira?

Resposta: De acordo com o contexto, você pode impulsionar sua carreira no universo corporativo se tornando um profissional altamente capacitado e desenvolvendo uma carreira na área de Gestão de Pessoas e Psicologia Organizacional e do Trabalho. Além disso, é mencionado que a disciplina de pós-graduação em Gestão de Pessoas em Cooperativas pode ser uma opção para fortalecer a colaboração e alcançar o sucesso coletivo.
-

KeyboardInterrupt: Interrupted by user